# CO₂ story — one graph per cell (upload edition)

**How to use:**
1. Run **Setup 1** (installs libraries) and **Setup 2** (defines the plotting functions).
2. Run **Setup 3** and, when prompted, **upload these 3 files**:
   `co2_per_capita.csv`, `percapita_co2_by_source.csv`, `us_co2_by_fuel.csv`
3. Then run any graph cell below — **each cell draws exactly one graph.**

## Setup 1 — install libraries

In [ ]:
!pip -q install matplotlib pandas numpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print('libraries ready ✓')

## Setup 2 — define the plotting functions (`viz_lib`)

In [ ]:
"""Shared visual identity for every plot in the library.

One place sets the palette, fonts, grid and spines so that a scatter plot and
a line plot drawn a year apart still look like they came from the same tool.

The categorical palette is the validated, colorblind-safe order from the
project's data-viz design system (worst adjacent CVD deltaE ~9, normal-vision
~20). Hues are assigned to series in fixed order and never cycled: past eight
series, fold the tail into "Other" or facet instead of inventing new colors.
"""


import matplotlib as mpl

#: Sequential "smog / heat" ramp for CO2 magnitude: pale haze -> deep ember.
#: Lightness decreases monotonically, so it stays legible in black & white and
#: for colorblind readers (darker always means more emissions).
SMOG: list[str] = ["#f4d06a", "#eaa23b", "#df6b2e", "#c0392b", "#7b1f16"]


def smog_color(value: float, vmax: float) -> tuple:
    """Map ``value`` (0..``vmax``) onto the SMOG ramp; returns an RGBA tuple.

    Colors magnitude, not identity: hotter = more CO2 per capita. Pass the same
    ``vmax`` to several charts so their colors share one honest scale.
    """
    import matplotlib.colors as mcolors

    cmap = mcolors.LinearSegmentedColormap.from_list("smog", SMOG)
    frac = 0.0 if vmax <= 0 else max(0.0, min(1.0, value / vmax))
    # start a little into the ramp so the smallest bars aren't near-white
    return cmap(0.12 + 0.88 * frac)


#: Categorical palette, light surface, in fixed assignment order.
PALETTE: list[str] = [
    "#2a78d6",  # 1 blue
    "#eb6834",  # 2 orange
    "#1baf7a",  # 3 aqua
    "#eda100",  # 4 yellow
    "#e87ba4",  # 5 magenta
    "#008300",  # 6 green
    "#4a3aa7",  # 7 violet
    "#e34948",  # 8 red
]

# Chrome / ink for the light surface these plots render on.
_INK_PRIMARY = "#0b0b0b"
_INK_SECONDARY = "#52514e"
_MUTED = "#898781"
_GRID = "#e1e0d9"
_AXIS = "#c3c2b7"
_SURFACE = "#fcfcfb"

_FONT_STACK = ["system-ui", "Segoe UI", "DejaVu Sans", "Arial", "sans-serif"]


def series_color(index: int) -> str:
    """Return the palette hue for the *index*-th series (0-based).

    Raises ``IndexError`` past the eighth slot rather than silently cycling a
    hue — a reused color is a correctness bug in a categorical chart. Callers
    with more than eight series should fold the tail into "Other" or facet.
    """
    if index < 0 or index >= len(PALETTE):
        raise IndexError(
            f"series index {index} out of range; the categorical palette has "
            f"{len(PALETTE)} slots. Fold extra series into 'Other' or facet."
        )
    return PALETTE[index]


def apply_theme() -> None:
    """Apply the library's rcParams globally.

    Idempotent — safe to call more than once. Called automatically the first
    time a plot function runs, so most users never call it directly.
    """
    mpl.rcParams.update(
        {
            "figure.facecolor": _SURFACE,
            "axes.facecolor": _SURFACE,
            "savefig.facecolor": _SURFACE,
            "font.family": "sans-serif",
            "font.sans-serif": _FONT_STACK,
            "font.size": 11,
            "text.color": _INK_PRIMARY,
            "axes.edgecolor": _AXIS,
            "axes.labelcolor": _INK_SECONDARY,
            "axes.titlecolor": _INK_PRIMARY,
            "axes.titlesize": 12.5,
            "axes.titleweight": "bold",
            "axes.linewidth": 1.0,
            "axes.grid": True,
            "axes.grid.axis": "y",
            "axes.spines.top": False,
            "axes.spines.right": False,
            "grid.color": _GRID,
            "grid.linewidth": 1.0,
            "xtick.color": _MUTED,
            "ytick.color": _MUTED,
            "xtick.labelsize": 10.5,
            "ytick.labelsize": 10.5,
            "axes.prop_cycle": mpl.cycler(color=PALETTE),
            "legend.frameon": False,
            "legend.fontsize": 10.5,
        }
    )


"""Ranked horizontal bar chart, built to the Evergreen Data Viz Checklist.

One job: take a tidy pandas DataFrame and draw a sorted, directly-labelled
horizontal bar chart that a non-technical reader understands at a glance.

Checklist choices baked in: bars sorted by value (not alphabetically), a zero
baseline, direct value labels, no gridlines / border / redundant axis, a
takeaway title, and a CO2-themed sequential color ramp that stays legible in
black & white and for colorblind readers.
"""


import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch


_UP, _DOWN = "▲", "▼"  # ▲ ▼
_SURFACE = "#fcfcfb"


def _readable_ink(color) -> str:
    """Pick black or white text for a filled segment by its luminance."""
    r, g, b = mcolors.to_rgb(color)
    lum = 0.299 * r + 0.587 * g + 0.114 * b
    return "#0b0b0b" if lum > 0.6 else "#ffffff"


def ranked_bar(
    df,
    category: str,
    value: str,
    *,
    compare: str | None = None,
    reference: float | None = None,
    reference_label: str | None = None,
    vmax: float | None = None,
    unit: str = "",
    value_fmt: str = "{:.1f}",
    title: str | None = None,
    subtitle: str | None = None,
    note: str | None = None,
    ascending: bool = False,
    ax=None,
    figsize: tuple[float, float] | None = None,
):
    """Draw a sorted horizontal bar chart from a pandas DataFrame.

    Parameters
    ----------
    df
        A pandas DataFrame (MVP input — DataFrames only).
    category, value
        Column names: the label per bar and the numeric length to rank by.
    compare
        Optional column with an earlier value; when given, each bar gets a
        muted ``▲ / ▼ %`` tag showing the change to ``value`` (the story of
        who rose or fell).
    reference, reference_label
        Optional vertical reference line (e.g. the world average) and its
        label — instant context for "how far above normal".
    vmax
        Upper bound for the color ramp. Pass the same ``vmax`` to several
        charts so a bar of a given darkness means the same emissions in each.
        Defaults to this chart's maximum.
    unit
        Unit string appended to the reference label / used in labels.
    value_fmt
        Format string for the value labels.
    title, subtitle, note
        Takeaway title, units/what-am-I-looking-at subtitle, and a source note.
    ascending
        Sort direction; default puts the largest bar on top.
    ax, figsize
        Optional target Axes and figure size (height auto-scales with bars).

    Returns
    -------
    matplotlib.figure.Figure
    """
    apply_theme()

    data = df[[category, value] + ([compare] if compare else [])].dropna(subset=[value])
    data = data.sort_values(value, ascending=ascending).reset_index(drop=True)
    labels = data[category].tolist()
    values = data[value].tolist()
    n = len(values)
    if n == 0:
        raise ValueError("no rows to plot after dropping missing values")

    top = vmax if vmax is not None else max(values)

    owns_fig = ax is None
    if owns_fig:
        if figsize is None:
            figsize = (7.6, 0.52 * n + 1.7)
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    # bars top-to-bottom (largest at top when ascending=False)
    y = list(range(n))[::-1]
    for yi, val in zip(y, values):
        ax.barh(yi, val, height=0.68, color=smog_color(val, top), zorder=3)

    xmax = max(values + ([reference] if reference else []))
    ax.set_xlim(0, xmax * 1.16)  # head-room for end labels

    # direct value labels (+ optional change tag) at each bar end
    for yi, (val, row) in zip(y, zip(values, data.itertuples(index=False))):
        label = value_fmt.format(val)
        ax.annotate(label, xy=(val, yi), xytext=(6, 0), textcoords="offset points",
                    va="center", ha="left", fontsize=11, fontweight="bold",
                    color="#0b0b0b")
        if compare:
            prev = getattr(row, compare) if hasattr(row, compare) else None
            if prev:
                pct = (val - prev) / prev * 100
                up = pct >= 0
                tag = f"  {_UP if up else _DOWN} {abs(pct):.0f}%"
                # up = more emissions (bad) = warm; down = good = green
                ax.annotate(tag, xy=(val, yi), xytext=(6 + 34, 0),
                            textcoords="offset points", va="center", ha="left",
                            fontsize=9.5, fontweight="bold",
                            color="#c0392b" if up else "#0a7d33")

    # optional reference line for context, labelled at the baseline
    if reference is not None:
        ax.axvline(reference, color="#52514e", linestyle=(0, (4, 3)),
                   linewidth=1.2, zorder=2)
        rlab = reference_label or "reference"
        val_txt = f"{value_fmt.format(reference)}{(' ' + unit) if unit else ''}"
        ax.annotate(f"{rlab} ({val_txt})", xy=(reference, -0.75),
                    xytext=(4, 0), textcoords="offset points",
                    va="center", ha="left", fontsize=9.5, style="italic",
                    color="#52514e")

    # category labels; strip every non-data line (checklist: mute the lines)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=11)
    ax.set_xticks([])
    for side in ("top", "right", "bottom", "left"):
        ax.spines[side].set_visible(False)
    ax.grid(False)
    ax.tick_params(length=0)
    # headroom above the top bar for the subtitle, a little below for the ref label
    ax.set_ylim(-1.1, n - 1 + 0.9)

    # titles: takeaway on top, quiet subtitle beneath, source note at the foot
    if title:
        ax.set_title(title, loc="left", fontsize=15, fontweight="bold", pad=24)
    if subtitle:
        ax.annotate(subtitle, xy=(0, 1.0), xycoords="axes fraction",
                    xytext=(0, 8), textcoords="offset points",
                    ha="left", va="bottom", fontsize=11, color="#52514e")
    if note:
        ax.annotate(note, xy=(0, 0), xycoords="axes fraction",
                    xytext=(0, -26), textcoords="offset points",
                    ha="left", va="top", fontsize=8.5, color="#898781")

    if owns_fig:
        fig.tight_layout()
    return fig


def stacked_bar(
    df,
    category: str,
    segments: list[str],
    *,
    colors=None,
    ascending: bool = False,
    unit: str = "",
    value_fmt: str = "{:.0f}",
    seg_label_min: float = 0.08,
    title: str | None = None,
    subtitle: str | None = None,
    note: str | None = None,
    legend: bool = True,
    ax=None,
    figsize: tuple[float, float] | None = None,
):
    """Draw a ranked, stacked horizontal bar chart from a pandas DataFrame.

    Each row is one category (e.g. a country); ``segments`` are the columns
    that stack into its total. Rows are ordered by total, the total is labelled
    at the bar end, and wide-enough segments are labelled in place — the same
    idea as the Our World in Data "by source" charts.

    Parameters
    ----------
    df
        A pandas DataFrame, one row per category.
    category
        Column holding the row label per bar.
    segments
        Columns to stack, left-to-right. Missing values count as zero.
    colors
        ``None`` assigns the categorical palette in order (segments are
        categories — identity, not magnitude), or a dict of overrides.
    ascending
        Sort direction by total; default puts the largest total on top.
    unit, value_fmt
        Number format for the labels. ``value_fmt`` may be a format string
        (``unit`` is appended) or a callable ``value -> str`` for adaptive
        formatting (e.g. one decimal below 10, none above); with a callable,
        ``unit`` is ignored and the callable owns the whole label.
    seg_label_min
        Only label a segment in place if it is at least this fraction of the
        largest row total (keeps small slivers uncluttered).
    title, subtitle, note
        Takeaway title, quiet subtitle, and source note.
    legend
        Draw a segment legend above the plot (default True).
    ax, figsize
        Optional target Axes and figure size (height auto-scales with rows).

    Returns
    -------
    matplotlib.figure.Figure
    """
    apply_theme()

    data = df[[category] + segments].copy()
    for s in segments:
        data[s] = data[s].fillna(0.0)
    data["_total"] = data[segments].sum(axis=1)
    data = data.sort_values("_total", ascending=ascending).reset_index(drop=True)
    n = len(data)
    if n == 0:
        raise ValueError("no rows to plot")

    overrides = dict(colors) if isinstance(colors, dict) else {}
    seg_colors = {s: overrides.get(s, series_color(i)) for i, s in enumerate(segments)}

    owns_fig = ax is None
    if owns_fig:
        if figsize is None:
            figsize = (9.0, 0.55 * n + 2.0)
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    # one formatter for both the in-segment labels and the bar-end total;
    # pass a callable for adaptive formatting (e.g. "6.4 t" but "34 t")
    if callable(value_fmt):
        fmt = value_fmt
    else:
        def fmt(v):
            return f"{value_fmt.format(v)}{(' ' + unit) if unit else ''}"

    y = list(range(n))[::-1]
    max_total = float(data["_total"].max())
    label_floor = max_total * seg_label_min

    for yi, (_, row) in zip(y, data.iterrows()):
        left = 0.0
        for s in segments:
            w = float(row[s])
            if w <= 0:
                continue
            color = seg_colors[s]
            ax.barh(yi, w, left=left, height=0.7, color=color, zorder=3,
                    edgecolor=_SURFACE, linewidth=1.0)
            if w >= label_floor:  # label only segments wide enough to fit
                ax.annotate(fmt(w), xy=(left + w / 2, yi),
                            ha="center", va="center", fontsize=9.5,
                            fontweight="bold", color=_readable_ink(color), zorder=4)
            left += w
        # total at the bar end
        ax.annotate(fmt(left), xy=(left, yi), xytext=(6, 0),
                    textcoords="offset points", va="center", ha="left",
                    fontsize=11, fontweight="bold", color="#0b0b0b")

    ax.set_xlim(0, max_total * 1.16)
    ax.set_ylim(-0.8, n - 1 + (1.4 if legend else 0.7))
    ax.set_yticks(y)
    ax.set_yticklabels(data[category].tolist(), fontsize=11)
    ax.set_xticks([])
    for side in ("top", "right", "bottom", "left"):
        ax.spines[side].set_visible(False)
    ax.grid(False)
    ax.tick_params(length=0)

    if legend:
        handles = [Patch(facecolor=seg_colors[s], label=s) for s in segments]
        ax.legend(handles=handles, loc="lower left", bbox_to_anchor=(0, 1.0),
                  ncol=min(len(segments), 6), frameon=False, fontsize=9.5,
                  handlelength=1.1, columnspacing=1.4, borderaxespad=0)

    if title:
        ax.set_title(title, loc="left", fontsize=15, fontweight="bold", pad=44)
    if subtitle:
        ax.annotate(subtitle, xy=(0, 1.0), xycoords="axes fraction",
                    xytext=(0, 26), textcoords="offset points", ha="left",
                    va="bottom", fontsize=11, color="#52514e")
    if note:
        ax.annotate(note, xy=(0, 0), xycoords="axes fraction", xytext=(0, -26),
                    textcoords="offset points", ha="left", va="top",
                    fontsize=8.5, color="#898781")

    if owns_fig:
        fig.tight_layout()
    return fig


"""Stacked area chart — composition (part-to-whole) over time.

One job: show how several series stack into a total across an ordered x-axis,
with direct band labels and optional dated event markers for a narrative,
editorial "hero" chart.
"""


import numpy as np
import matplotlib.pyplot as plt


_MUTED = "#898781"
_INK = "#0b0b0b"
_SECOND = "#52514e"
_SURFACE = "#fcfcfb"


def stacked_area(
    df,
    x: str,
    series: list[str],
    *,
    colors=None,
    y_label: str | None = None,
    title: str | None = None,
    subtitle: str | None = None,
    note: str | None = None,
    events: list[dict] | None = None,
    direct_labels: bool = True,
    ax=None,
    figsize: tuple[float, float] | None = None,
):
    """Draw a stacked area chart from a pandas DataFrame.

    Parameters
    ----------
    df
        A pandas DataFrame.
    x
        Column name for the (ordered) x-axis, e.g. ``"Year"``.
    series
        Column names to stack, in **bottom-to-top** order. Missing values are
        treated as zero (so a band simply starts once its data begins).
    colors
        ``None`` assigns the categorical palette in order (fuels are
        categories, so identity — not magnitude — drives color), or pass a
        dict of ``{series_name: color}`` to override.
    y_label, title, subtitle, note
        Axis label, takeaway title, quiet subtitle, and source note.
    events
        Optional list of ``{"year": int, "label": str, "y": float}`` markers.
        ``y`` (0–1, default 0.95) sets the label height as a fraction of the
        axis; each draws a thin vertical rule + label for a dated annotation.
    direct_labels
        Label each band at its right end instead of a legend (default True).
    ax, figsize
        Optional target Axes and figure size.

    Returns
    -------
    matplotlib.figure.Figure
    """
    apply_theme()

    data = df.sort_values(x)
    xv = data[x].to_numpy(dtype=float)
    stacks = [np.nan_to_num(data[s].to_numpy(dtype=float), nan=0.0) for s in series]
    overrides = dict(colors) if isinstance(colors, dict) else {}
    cols = [overrides.get(s, series_color(i)) for i, s in enumerate(series)]

    owns_fig = ax is None
    if owns_fig:
        fig, ax = plt.subplots(figsize=figsize or (11, 6))
    else:
        fig = ax.figure

    # 2px surface gap between bands (checklist: separate the fills)
    ax.stackplot(xv, *stacks, colors=cols, edgecolor=_SURFACE, linewidth=0.8)

    ax.set_xlim(xv.min(), xv.max())
    top = np.sum(stacks, axis=0).max()
    ax.set_ylim(0, top * 1.02)

    # always mark the final year (e.g. 2024) as a tick, like the OWID charts
    ticks = [t for t in ax.get_xticks() if xv.min() <= t <= xv.max()]
    if not ticks or xv.max() - ticks[-1] > 6:
        ticks.append(xv.max())
    else:
        ticks[-1] = xv.max()  # snap a too-close tick onto the exact end
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{int(t)}" for t in ticks])

    # direct band labels at the right end, nudged apart if they collide
    if direct_labels:
        cum = np.cumsum(stacks, axis=0)
        centers = []
        for i, s in enumerate(series):
            bottom = cum[i - 1][-1] if i else 0.0
            centers.append(((bottom + cum[i][-1]) / 2, s, cols[i]))
        _labels_at_right(ax, xv.max(), centers, top)

    # dated event markers (the editorial "timeline" layer)
    x_lo, x_hi = xv.min(), xv.max()
    near_right = x_lo + 0.85 * (x_hi - x_lo)
    for ev in events or []:
        yr = ev["year"]
        ax.axvline(yr, color=_MUTED, linewidth=1.0, linestyle=(0, (2, 2)), zorder=5)
        yfrac = ev.get("y", 0.95)
        # flip the label to the left of its rule near the right edge, so it
        # never lands on top of the right-hand band labels
        right = yr >= near_right
        ax.annotate(ev["label"], xy=(yr, top * yfrac),
                    xytext=(-4 if right else 4, 0), textcoords="offset points",
                    va="top", ha="right" if right else "left",
                    fontsize=9, color=_SECOND, zorder=6,
                    bbox=dict(boxstyle="round,pad=0.15", fc=_SURFACE, ec="none", alpha=0.85))

    # chrome: mute everything that isn't data (checklist: mute the lines)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.spines["left"].set_color(_MUTED)
    ax.spines["bottom"].set_color(_MUTED)
    ax.tick_params(length=0)
    ax.grid(False)
    if y_label:
        ax.set_ylabel(y_label, color=_SECOND)

    if title:
        ax.set_title(title, loc="left", fontsize=17, fontweight="bold", pad=26)
    if subtitle:
        ax.annotate(subtitle, xy=(0, 1.0), xycoords="axes fraction",
                    xytext=(0, 8), textcoords="offset points", ha="left",
                    va="bottom", fontsize=11.5, color=_SECOND)
    if note:
        ax.annotate(note, xy=(0, 0), xycoords="axes fraction", xytext=(0, -34),
                    textcoords="offset points", ha="left", va="top",
                    fontsize=8.5, color=_MUTED)

    if owns_fig:
        fig.tight_layout()
    return fig


def _labels_at_right(ax, x_end, centers, top):
    """Place band labels at the right edge, spread so they don't overlap."""
    gap = top * 0.045
    centers = sorted(centers, key=lambda t: t[0])
    prev = None
    for y_val, name, color in centers:
        y_text = y_val if prev is None else max(y_val, prev + gap)
        prev = y_text
        ax.annotate(name, xy=(x_end, y_text), xytext=(8, 0),
                    textcoords="offset points", va="center", ha="left",
                    fontsize=10.5, fontweight="bold", color=color,
                    annotation_clip=False)

print('plotting functions ready ✓')

## Setup 3 — upload the 3 CSV files
Run this cell, click **Choose Files**, and select the three CSVs.

In [ ]:
from google.colab import files
print('Upload: co2_per_capita.csv, percapita_co2_by_source.csv, us_co2_by_fuel.csv')
uploaded = files.upload()
print('uploaded:', list(uploaded))

## Graph 1 — Oil producers: CO₂ per person
Uses `co2_per_capita.csv`.

In [ ]:
df = pd.read_csv('co2_per_capita.csv')
df = df[df.Year.isin([2014, 2024])].pivot_table(
        index='Entity', columns='Year', values='CO₂ emissions per capita')
df.columns = [f'y{c}' for c in df.columns]
df = df.reset_index()

OIL  = ['Qatar','Kuwait','Brunei','Bahrain','Trinidad and Tobago',
        'Saudi Arabia','United Arab Emirates','Oman']
ECON = ['United States','Russia','North America','China',
        'European Union (27)','World','United Kingdom','India']
vmax  = df[df.Entity.isin(OIL + ECON)].y2024.max()   # shared color scale
world = df.loc[df.Entity == 'World', 'y2024'].iloc[0]

ranked_bar(df[df.Entity.isin(OIL)], category='Entity', value='y2024', compare='y2014',
           vmax=vmax, unit='t', reference=world, reference_label='World average',
           title='A few small, oil-rich nations emit the most CO₂ per person',
           subtitle='Tonnes of CO₂ per person, 2024')
plt.show()

## Graph 2 — Major economies: CO₂ per person
Uses `co2_per_capita.csv` (same color scale as Graph 1).

In [ ]:
df = pd.read_csv('co2_per_capita.csv')
df = df[df.Year.isin([2014, 2024])].pivot_table(
        index='Entity', columns='Year', values='CO₂ emissions per capita')
df.columns = [f'y{c}' for c in df.columns]
df = df.reset_index()

OIL  = ['Qatar','Kuwait','Brunei','Bahrain','Trinidad and Tobago',
        'Saudi Arabia','United Arab Emirates','Oman']
ECON = ['United States','Russia','North America','China',
        'European Union (27)','World','United Kingdom','India']
vmax  = df[df.Entity.isin(OIL + ECON)].y2024.max()

ranked_bar(df[df.Entity.isin(ECON)], category='Entity', value='y2024', compare='y2014',
           vmax=vmax, unit='t',
           title='Among big economies, the US still emits the most per person',
           subtitle='Tonnes of CO₂ per person, 2024 — same scale as the oil producers')
plt.show()

## Graph 3 — Per capita CO₂ by source, 2024
Uses `percapita_co2_by_source.csv` (a replica of the Our World in Data chart).

In [ ]:
d = pd.read_csv('percapita_co2_by_source.csv')
SEG = ['Coal','Oil','Gas','Flaring','Cement','Other industry']
OWID = {'Coal':'#6d6e70','Oil':'#c14b62','Gas':'#8c6bb1',
        'Flaring':'#c8a45c','Cement':'#2f8e7f','Other industry':'#6d8fc5'}
tonnes = lambda v: f'{v:.0f} t' if v >= 10 else f'{v:.1f} t'

stacked_bar(d, category='Entity', segments=SEG, colors=OWID,
            value_fmt=tonnes, seg_label_min=0.05,
            title='Per capita CO₂ emissions by source, 2024',
            figsize=(11, 8))
plt.show()

## Graph 4 — US CO₂ by fuel over time (the hero)
Uses `us_co2_by_fuel.csv`.

In [ ]:
h = pd.read_csv('us_co2_by_fuel.csv')
FUELS = ['Coal','Oil','Gas','Cement','Flaring','Other industry']
for f in FUELS:
    h[f] = pd.to_numeric(h[f], errors='coerce') / 1e9   # tonnes -> billion tonnes

EVENTS = [{'year':1932,'label':'1932\nGreat Depression','y':0.42},
          {'year':1945,'label':'1945\nWWII','y':0.72},
          {'year':1973,'label':'1973\nOil shock','y':0.9},
          {'year':2007,'label':'2007\nemissions peak','y':0.98},
          {'year':2020,'label':'2020\nCOVID','y':0.62}]

stacked_area(h, x='Year', series=FUELS, y_label='Billion tonnes CO₂ / year',
             title='Coal gave way to oil and gas',
             subtitle='US CO₂ emissions by fuel or industry, 1800–2024',
             events=EVENTS)
plt.show()

---
*Each graph is drawn by a `viz_lib` function defined in Setup 2, from the CSV you uploaded in Setup 3.*